In [2]:
import sys
import pandas as pd
import numpy as np
import shap
from xgboost import XGBRegressor
import lightgbm as lgb
from sklearn.metrics import r2_score
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

In [10]:
def create_surrogate(X, predictions, sur_parameters):
    # Create surrogate xgboost model
    surrogate = XGBRegressor(**sur_parameters)
    surrogate.fit(X, predictions)
    sur_predictions = surrogate.predict(X)
    external_fidelity = r2_score(predictions, sur_predictions)
    return surrogate, external_fidelity

def calc_shap(model, x_explain, approximate=False):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(x_explain, approximate=approximate)
    return shap_values, explainer

In [18]:
from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = load_diabetes(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, shuffle=True)
original_model = lgb.LGBMRegressor()
original_model.fit(X_train, y_train, eval_set = [(X_val,y_val)],early_stopping_rounds = 5, eval_metric = 'rmse')
predictions = original_model.predict(X)

# Use agnostic explainer
sur_parameters = {'n_estimators': 100, 'max_depth': 8}
surrogate_model, external_fidelity = create_surrogate(X,
                                                predictions,
                                                sur_parameters)
# get the shapley values for the whole dataset
shap_values, explainer = calc_shap(surrogate_model, X)

'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.


[1]	valid_0's rmse: 76.5308	valid_0's l2: 5856.97
[2]	valid_0's rmse: 73.4823	valid_0's l2: 5399.64
[3]	valid_0's rmse: 71.1803	valid_0's l2: 5066.63
[4]	valid_0's rmse: 69.1122	valid_0's l2: 4776.5
[5]	valid_0's rmse: 67.2882	valid_0's l2: 4527.7
[6]	valid_0's rmse: 65.6426	valid_0's l2: 4308.95
[7]	valid_0's rmse: 64.3669	valid_0's l2: 4143.09
[8]	valid_0's rmse: 63.2836	valid_0's l2: 4004.82
[9]	valid_0's rmse: 62.4004	valid_0's l2: 3893.81
[10]	valid_0's rmse: 61.751	valid_0's l2: 3813.19
[11]	valid_0's rmse: 60.8987	valid_0's l2: 3708.65
[12]	valid_0's rmse: 60.5354	valid_0's l2: 3664.53
[13]	valid_0's rmse: 59.765	valid_0's l2: 3571.85
[14]	valid_0's rmse: 59.4965	valid_0's l2: 3539.83
[15]	valid_0's rmse: 59.1177	valid_0's l2: 3494.91
[16]	valid_0's rmse: 58.7745	valid_0's l2: 3454.45
[17]	valid_0's rmse: 58.535	valid_0's l2: 3426.34
[18]	valid_0's rmse: 58.0553	valid_0's l2: 3370.42
[19]	valid_0's rmse: 57.9245	valid_0's l2: 3355.25
[20]	valid_0's rmse: 57.9253	valid_0's l2: 33

ntree_limit is deprecated, use `iteration_range` or model slicing instead.


In [19]:

iteration_number = 2
vals= np.abs(shap_values).mean(0)

X_train = pd.DataFrame(X_train)

feature_importance = pd.DataFrame(list(zip(X_train.columns,vals)),columns=['col_name','feature_importance_vals'])
feature_importance.sort_values(by=['feature_importance_vals'],ascending=False,inplace=True)
feature_importance["col_name"] =  'Feature ' + feature_importance["col_name"].astype(str)

feature_importance = feature_importance.head()

feature_importance.to_csv(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt', header=None, index=None, sep=' ', mode='a')

with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','a') as file:
    file.write("\nRun number: %d\n\n" %iteration_number)
file.close()


In [20]:
experiments=2

In [21]:
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','r') as file:
    data = file.readlines()
file.close()

#-----------------------------------------------------------------------------------------------------------------
    
# Make script that compares the  data - explanations, and the highlighted features (if top feature is the same)

# This is a function for getting the change % of a feature (will not be used here)
def get_change(current, previous):
    
    if current == previous:
        return 100.0
    try:
        change = (abs(current - previous) / ((current + previous)/2) ) * 100.0
        return "{:.2f}".format(change)
    except ZeroDivisionError:
        return 0

values = []
features = []
for i in range(experiments):
    for j in range(5):
        x = data[i*8+j].split()
        values.append(x[2])
        features.append(x[0]+x[1])

# Make changes here on the dimensions of the dataframes for evaluating more experiments
        
# Make dataframe with the shapley values of the features for each run  
values = np.array(values)
sub_arrays = np.hsplit(values, experiments)
df = pd.DataFrame(sub_arrays)
for i in range(experiments):
    df.index = ['Run_%d'%(i+1) for i in range(experiments)]
df.columns = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']

# Make dataframe with the feature names (ranking of the features at each experiment)
features = np.array(features)
sub_arrays2 = np.hsplit(features, experiments)
df2 = pd.DataFrame(sub_arrays2)
for i in range(experiments):
    df2.index = ['Run_%d'%(i+1) for i in range(experiments)]
df2.columns = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']
df2 = df2.applymap(lambda x: x.replace('"', ''))

# Measure the difference of the 1st feature between the different runs (if the top feature is the same)

# if df2.loc["Run_1","1st Feature"] == df2.loc["Run_2","1st Feature"]:
#     print(get_change(float(df.loc["Run_1","1st Feature"]),float(df.loc["Run_2","1st Feature"])),'%')
# else:
#     print(df2)

# Gather here the unique features
uniqueValues = (df2['1st Feature'].append(df2['2nd Feature'].append(df2['3rd Feature'].append(df2['4th Feature'].append(df2['5th Feature']))))).unique()

df2 = df2.astype(str)
print("\nOur dataframes are:\n\n",df,"\n\n",df2,"\n")  
print("\nDescribe df with the feature names:\n\n", df2.describe())
print("\n..........................................\n")    

#-----------------------------------------------------------------------------------------------------------------

# Make a dictionary for storing the unique feature names as keys and their corresponding shapley values as values

df1 = df.astype(float)
dict = {animal: df1[df2.eq(animal)].stack().values for animal in np.unique(df2)}
print("\nThis is the new dictionary we created:\n", dict)

# Calculate std deviation for each feature

matrix = []
matrix2 = []
matrix3= []
matrix4 = []

# In matrix we will store the keys (features) and in matrix2 the values (shapley values)
for i in dict:
    matrix.append(i)
    matrix2.append(dict[i])

# In matrix3 we calculate the standard deviation of the features 
for list in matrix2:
    matrix3.append("{:.2f}".format(np.std(list, dtype=np.float64)))

# In matrix4 we calculate the relative standard deviation (percentage)
for list in matrix2:
    matrix4.append(str("{:.2f}".format(100*np.std(list, dtype=np.float64)/len(list)))+"%")
    
print("\nThis is the matrix with the unique features: \n", *matrix, sep = '\n')
print("\nThis is the matrix with the list of values each feature has: \n", *matrix2, sep = '\n')   
print("\nThis is the the std deviation of each list: \n", *matrix3, sep = '\n')  
print("\nThis is the relative std deviation of each list of values: \n", *matrix4, sep = '\n')

# Function for returning a list of the keys of a dictionary

def getList(dict):
    feature_names = []
    for key in dict.keys():
        feature_names.append(key)
          
    return feature_names

keys = getList(dict) 

del list

# Concatenate the lists of the features and the deviations
std_deviation_of_features = list(zip(keys, matrix3))
re_std_deviation_of_features = list(zip(keys, matrix4))

print("\nThe standard deviation of our features is: \n", *std_deviation_of_features, sep = '\n')
print("\nThe relative standard deviation of our features is: \n", *re_std_deviation_of_features, sep = '\n')
print("\n..........................................\n")    

#-----------------------------------------------------------------------------------------------------------------

# Measuring the ranking deviation of the features

# Function returning the index positions of a df where an element is present
def getIndexes(dfObj, value):
     
    # Empty list
    listOfPos = []
     
    # isin() method will return a dataframe with
    # boolean values, True at the positions   
    # where element exists
    result = dfObj.isin([value])
     
    # any() method will return
    # a boolean series
    seriesObj = result.any()
 
    # Get list of column names where
    # element exists
    columnNames = list(seriesObj[seriesObj == True].index)
    
    # Iterate over the list of columns and
    # extract the row index where element exists
    for col in columnNames:
        rows = list(result[col][result[col] == True].index)
 
        for row in rows:
            listOfPos.append((row, col))
             
    # This list contains a list tuples with
    # the index of element in the dataframe
    return listOfPos
 
# Calling getIndexes() function 

listOfPositions = []
for item in uniqueValues:
    listOfPositions.append(str(item) + str(getIndexes(df2, item)))   
    
print("\nThese are the positions where each feature is present and what ranking it has:\n ",*listOfPositions, sep = '\n')  

# In how many columns the feature occurs
# len(listOfPositions[5].split(', ')) / 2

listOfPositions2 = {}
for item in uniqueValues:
    listOfPositions2[item] = getIndexes(df2, item)
    
# for item in uniqueValues:
#     print(listOfPositions2["%s"%item])

# c = df2.columns[(df2 == 'Feature23').any()].tolist()
# d = df2.value_counts()

# matrix = []
# for col in df2.columns:
#     matrix.append(df2[col].value_counts())

# Magic matrix contains a list of integers reflecting the positions where each feature is present in the ranking
magic = []
keys = []
rankings = {}
for item in uniqueValues:
    new = []
    for col in df2.columns:
        for row in df2.index:
            if df2.at[row,col] == item:
                new.append(int(str(col).split(' ')[0][0]))
    magic.append(new)

# The rankings array will contain the std deviation of the rankings measured before
rankings = []
for list in magic:
    rankings.append(np.std(list, dtype=np.float64))

# The ranking_deviation concatenates the unique features with the std deviation into a comprehensive list
ranking_deviation = []
for i in range(len(uniqueValues)):
    ranking_deviation.append(uniqueValues[i]+": "+str(rankings[i]))

print("\nThe ranking deviation of each feature is: \n", *ranking_deviation, sep = '\n')

rankings_percentage = []
for list in magic:
    rankings_percentage.append(str("{:.2f}".format(100*np.std(list, dtype=np.float64)/len(list)))+"%")
    
ranking_deviation_percentage = []
for i in range(len(uniqueValues)):
    ranking_deviation_percentage.append(uniqueValues[i]+": "+str(rankings_percentage[i]))   
    
print("\nThis is the ranking deviation percentage for each feature: \n",*ranking_deviation_percentage, sep = '\n')
print("\n..........................................\n")    
    
# Research ranking deviation and visualize other statistics (distribution)   
# Expand on more runs and features (also on scenario 2)

# Save our dataframes for reference
df1.to_csv(r'df1.csv', index = False)
df2.to_csv(r'df2.csv', index = False)

# Calculate the mean of the ranking deviation

rankings_percentage_float = []
for alist in magic:
    rankings_percentage_float.append(float("{:.2f}".format(100*np.std(alist, dtype=np.float64)/len(alist))))
    
final_result = str("{:.2f}".format(np.mean(rankings_percentage_float)))

# Print the mean of the ranking deviation
print("Mean of the ranking deviation: "+final_result+"%")

# Calculate the mean of the std deviation
std_deviation_float = []
for alist in matrix2:
    std_deviation_float.append(float("{:.2f}".format(100*np.std(alist, dtype=np.float64)/len(alist))))
    
final_result2 = str("{:.2f}".format(np.mean(std_deviation_float)))

# Print the mean of the std deviation
print("\nMean of the standard deviation (from importance of features): "+final_result2+"%")


Our dataframes are:

       1st Feature 2nd Feature 3rd Feature 4th Feature 5th Feature
Run_1   31.252855   18.792677    6.831917   5.9881544   3.8340862
Run_2   34.349472    19.83386   5.9005523    4.599955   3.8389614 

       1st Feature 2nd Feature 3rd Feature 4th Feature 5th Feature
Run_1    Feature8    Feature2    Feature3    Feature6    Feature9
Run_2    Feature8    Feature2    Feature3    Feature1    Feature9 


Describe df with the feature names:

        1st Feature 2nd Feature 3rd Feature 4th Feature 5th Feature
count            2           2           2           2           2
unique           1           1           1           2           1
top       Feature8    Feature2    Feature3    Feature6    Feature9
freq             2           2           2           1           2

..........................................


This is the new dictionary we created:
 {'Feature1': array([4.599955]), 'Feature2': array([18.792677, 19.83386 ]), 'Feature3': array([6.831917 , 5.9005523])

In [1]:
import sys
import pandas as pd
import numpy as np
import shap
import statistics
from xgboost import XGBRegressor
import lightgbm as lgb
from sklearn.metrics import r2_score
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
import os

In [19]:
def create_surrogate(X, predictions, sur_parameters):
    # Create surrogate xgboost model
    surrogate = XGBRegressor(**sur_parameters, random_state=1)
    surrogate.fit(X, predictions)
    sur_predictions = surrogate.predict(X)
    external_fidelity = r2_score(predictions, sur_predictions)
    return surrogate, external_fidelity

def create_surrogate2(X, predictions, sur_parameters):
    # Create surrogate xgboost model
    surrogate = XGBRegressor(**sur_parameters)
    surrogate.fit(X, predictions)
    sur_predictions = surrogate.predict(X)
    external_fidelity = r2_score(predictions, sur_predictions)
    return surrogate, external_fidelity

def calc_shap(model, x_explain, approximate=False):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(x_explain, approximate=approximate)
    return shap_values, explainer

from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = load_diabetes(return_X_y=True)

X_train, _, y_train, _ = train_test_split(X, y, shuffle=False)
original_model = MLPClassifier(random_state=1)
original_model.fit(X_train, y_train)
predictions = original_model.predict(X)

Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.


In [20]:
# Use agnostic explainer
sur_parameters = {'n_estimators': 100, 'max_depth': 8}
surrogate_model1, external_fidelity = create_surrogate(X,
                                                predictions,
                                                sur_parameters)

In [21]:
# Use agnostic explainer
sur_parameters = {'n_estimators': 100, 'max_depth': 8}
surrogate_model2, external_fidelity = create_surrogate2(X,
                                                predictions,
                                                sur_parameters)

In [22]:
(surrogate_model1.predict(X) == surrogate_model2.predict(X)).all()

True

In [23]:
external_fidelity

0.9999999937952924

In [2]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

def calc_shap(model, x_explain):
    explainer = shap.KernelExplainer(model.predict_proba, X_train)        
    shap_values = explainer.shap_values(x_explain)
    return shap_values, explainer

from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = make_classification(n_samples=50000, n_features=30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = MLPClassifier()
model.fit(X_train, y_train)

C:\Users\Stefanos Rigas\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier()

In [14]:
X_train = pd.DataFrame(X_train)
feature_names = X_train.columns
feature_names

shap_values = [[30]*30]

In [15]:
import numpy as np
vals= np.abs(shap_values).mean(0)

X_train = pd.DataFrame(X_train)

feature_importance = pd.DataFrame(list(zip(X_train.columns,vals)),columns=['col_name','feature_importance_vals'])
feature_importance.sort_values(by=['feature_importance_vals'],ascending=False,inplace=True)
feature_importance["col_name"] =  'Feature ' + feature_importance["col_name"].astype(str)

feature_importance = feature_importance.head()
feature_importance

,col_name,feature_importance_vals
0,Feature 0,30.0
1,Feature 1,30.0
28,Feature 28,30.0
27,Feature 27,30.0
26,Feature 26,30.0
